# African elections frame analysis

*Do international media outlets frame African elections differently from African outlets — and if so, along which dimensions?*

**Primary deliverable notebook.** This notebook is the main analytical narrative. Pipeline modules live in `src/elections_frames/`; the evaluation loop lives in `04_pipeline_eval.ipynb`; sensitivity checks live in `03_robustness.ipynb`.

Every meaningful data-processing decision in this notebook follows the five-part block in `NOTEBOOK_STRUCTURE.md`.

## 1. The question

Framing analysis — *which dimensions of African political life get foregrounded in international vs. African coverage of the same elections* — has lived mostly in essay form. This notebook quantifies it for four recent elections (Nigeria 2023, Kenya 2022, Senegal 2024, South Africa 2024) using a six-frame taxonomy (security / economy / democracy / identity / process / corruption).

**What this is not.** Not sentiment analysis. Not an evaluation of coverage *quality* or *accuracy*. Frame analysis is descriptive: *which dimensions are foregrounded*.

## 2. Data

- **GDELT 2.0 GKG** — article-level records, pulled ±30 days around each election. Coverage is whatever GDELT indexes; systematic bias toward publicly crawlable English-language sources (see Limitations).
- **Outlet provenance** — `data/external/outlets.csv`, hand-curated. African vs. International vs. Edge case (BBC Africa service, Al Jazeera English Africa desk, etc.), with verdicts resolved in the outlet-attribution decision block below.
- **Hand-labeled eval set** — `data/external/eval_set.parquet`, 200–300 articles stratified by election × outlet origin × week, **hand-labeled by Muhanad** (anti-anchoring: no LLM suggestions visible). Treated as immutable ground truth.

*Loading code added in Session 2.*

## 3. Ingestion status

Row counts per election, count of unknown-origin outlets — sanity output. (Populated in Session 2.)

In [ ]:
"""Ingestion status — row counts per election + unknown-origin outlet count.

This cell reads from local cache only — it does not pull. To populate the four
±30-day windows, run from the project root (multi-hour, multi-GB):

    python scripts/pull_all.py

The cell will produce empty rows for any election not yet pulled; that's
expected during early Session 2.
"""
from elections_frames.data import ELECTIONS, load_cached, outlet_provenance_join

rows = []
for key, election in ELECTIONS.items():
    df = load_cached(key)
    if len(df) == 0:
        rows.append({
            "election": election.name,
            "rows": 0,
            "unique_outlets": 0,
            "unknown_origin_rows": 0,
            "cached": False,
        })
        continue
    joined = outlet_provenance_join(df)
    rows.append({
        "election": election.name,
        "rows": len(joined),
        "unique_outlets": joined["SourceCommonName"].nunique(),
        "unknown_origin_rows": int((joined["outlet_origin"] == "Unknown").sum()),
        "cached": True,
    })

import pandas as pd
status = pd.DataFrame(rows)
status

## 4. Cleaning & structuring

This is where decision discipline lives most heavily — three decision blocks follow.

*(Each section below is a placeholder for a full five-part decision block; populated in Session 3.)*

### Decision 1: Outlet origin attribution

**Problem.** The headline question — *"do international and African outlets frame African elections differently?"* — only makes sense if we can attribute every article to one of those two buckets. GDELT's `SourceCommonName` is just the bare domain (`bbc.com`, `nation.africa`, `iheart.com`); we need a rule that converts ~5,000–6,000 unique outlets per election into a clean African/International label, including verdicts for the four named edge cases from `data/external/outlets.csv` (BBC Africa, Al Jazeera English Africa desk, Reuters Africa, RFI Afrique).

**Diagnostic.** Load a strided probe (one 15-min slot per day per election → ~244 zips total, ~320K rows pre-relevance-filter); count outlets, inspect URL patterns for the four named edge cases, and check the size of the unmatched tail.

In [ ]:
"""Probe-level diagnostic: outlet distribution and origin attribution.

Loads a strided sample of the cached GKG firehose (stride=96 → ~1 zip/day per
election) and applies `attribute_outlet_origin`. Cached to
`.cache/probe_attributed.parquet` on first run so re-running this cell is cheap.
"""
from pathlib import Path
import pandas as pd
from elections_frames.cleaning import load_probe, attribute_outlet_origin
from elections_frames.data import ELECTIONS

probe_cache = Path("../.cache/probe_attributed.parquet")

if probe_cache.exists():
    probe = pd.read_parquet(probe_cache)
else:
    frames = [load_probe(k, stride=96, progress=False) for k in ELECTIONS]
    probe = pd.concat(frames, ignore_index=True)
    probe = attribute_outlet_origin(probe)
    probe_cache.parent.mkdir(parents=True, exist_ok=True)
    probe.to_parquet(probe_cache, index=False)

print(f"Probe rows: {len(probe):,}")
print(f"Unique outlets: {probe.SourceCommonName.nunique():,}")
print()
print("Origin breakdown:")
print(probe.outlet_origin.value_counts().to_string())
print()
print("Top 10 outlets among International:")
print(probe[probe.outlet_origin == "International"].SourceCommonName.value_counts().head(10).to_string())
print()
print("Top 10 outlets among African:")
print(probe[probe.outlet_origin == "African"].SourceCommonName.value_counts().head(10).to_string())

In [ ]:
"""Edge-case URL inspection — does the URL actually distinguish the four named
African desks (BBC Africa, AJE Africa, Reuters Africa, RFI Afrique)?"""

def inspect_urls(probe: pd.DataFrame, domain_match: str, n: int = 5) -> list[str]:
    mask = probe.SourceCommonName.fillna("").str.lower().str.contains(domain_match)
    return probe.loc[mask, "DocumentIdentifier"].dropna().head(n).tolist()

for label, domain in [
    ("BBC", "bbc.com"),
    ("Al Jazeera English", "aljazeera.com"),
    ("Reuters", "reuters.com"),
    ("RFI", "rfi.fr"),
]:
    print(f"--- {label} ({domain}) — first 5 URLs ---")
    for u in inspect_urls(probe, domain):
        print(f"  {u}")
    print()

**What the diagnostic showed.**

- **Distribution is dominated by International** (≈98.9% pre-relevance-filter). The probe pulls the *global* GKG firehose; without an election-relevance filter the firehose is mostly global aggregators (`iheart.com`, `yahoo.com`, `biztoc.com`, `msn.com`) reposting wire copy unrelated to the four elections. The African slice (~1.1%) is dominated by recognizable outlets (`punchng.com`, `nation.africa`, `thisdaylive.com`, `dailymaverick.co.za`, etc.). This shape is the *firehose's* shape, not the *election coverage's* shape — the relevance filter (Decision 2) is what aligns the corpus with the question.
- **Only BBC reliably encodes Africa in the URL.** BBC Africa articles use slugs like `bbc.com/news/world-africa-NNNN` (hyphenated, *not* slash-separated, contrary to the original hypothesis in `outlets.csv`). AJE indexes articles by date (`aljazeera.com/news/2023/2/1/...`), Reuters by topic (`/business/...`, `/markets/...`), and RFI English uses `/en/africa/...` — but in the probe RFI hardly appears at all. **The path-sensitive rule works for BBC and (where present) RFI Afrique; it cannot reliably separate AJE-Africa or Reuters-Africa coverage from those outlets' main feeds.**

**Options considered.**

- **(a) URL-path matching where reliable; publisher-origin elsewhere.** BBC `world-africa` URLs → `Edge_BBC_Africa`; RFI `fr/afrique` and `en/africa` → `Edge_RFI_Afrique`; everything else by host (TLD chain + curated African-outlet list). All `aljazeera.com` and `reuters.com` rows → `International` per publisher origin, since URL doesn't disambiguate.
- **(b) Code all four named outlets as Edge.** Treat the whole of `bbc.com`, `aljazeera.com`, `reuters.com`, `rfi.fr` as a single `Edge` bucket regardless of URL, with a publisher-International verdict for the headline.
- **(c) Drop the four edge outlets entirely.** Cleaner cut, but loses signal from non-trivially-large publishers in this corpus.

**Decision.** **(a) URL-path matching where the URL supports it; publisher-origin classification elsewhere.** The portfolio question is *publisher-origin × framing*, so the default for all four named outlets is correctly `International`. We retain `Edge_BBC_Africa` / `Edge_RFI_Afrique` as labels (rolled into `International` for the headline analysis) so that the desk-staffing alternative interpretation can be tested in `03_robustness.ipynb` simply by toggling `collapse_edge_to_international` off. Verdicts and rationale are written into `data/external/outlets.csv`.

**Sensitivity.** The desk-staffing alternative — re-folding `Edge_*` labels into `African` — is the robustness check in `03_robustness.ipynb`. Expected impact on the headline figure is small: the probe shows `Edge_BBC_Africa` accounts for ~5 rows in a 320K-row probe (≈0.002%) — the absolute volume is dwarfed by both the African and International blocks. Whether this number grows after relevance filtering will be checked in Decision 2's `before_after`.

### Decision 2: Article-relevance filter

**Problem.** GDELT's GKG firehose is global — only ~0.05–0.7% of the rows we pulled actually concern one of the four elections. We need a rule that keeps election-relevant articles and drops the rest. False positives dilute the framing signal (an article about avocado prices kept under "South Africa" pollutes the South African block); false negatives shrink the analytic sample. We have three candidate rules: GDELT theme tags, election-specific keywords, or both (hybrid AND).

**Diagnostic.** Run all three methods against the probe and inspect a 10-row random sample of each method's output for the smallest-recall election. **One iteration was needed during this diagnostic:** the initial keyword set used substring matching on short tokens like `"DA"`, `"MK"`, `"EFF"`, `"Ba"` — which matched 40–70% of the firehose because of substring collisions (`avocado`, `bafana`, `effort`, `obama`). Fixing to **word-boundary regex** (`\b...\b`) collapsed false positives to a manageable rate. We also broadened the keyword list from 5–7 tokens per election to 10–14 (candidate first names, party long-forms, electoral commission names) after the initial run under-recalled in Senegal and Kenya.

In [ ]:
"""Three relevance methods compared on the probe (final post-iteration version)."""
from elections_frames.cleaning import filter_relevant
from elections_frames.diagnostics import compare_alternatives

rows = []
for key in ELECTIONS:
    sub = probe[probe.election == key]
    for method in ("theme", "keyword", "hybrid"):
        _, counts = filter_relevant(sub, ELECTIONS[key], method=method)
        rows.append({
            "election": key,
            "method": method,
            "n_in": counts["n_in"],
            "n_kept": counts["n_kept"],
            "pct_kept": round(100 * counts["n_kept"] / counts["n_in"], 2),
        })
rel_summary = pd.DataFrame(rows).pivot_table(
    index="election", columns="method", values="n_kept", aggfunc="sum"
)[["theme", "keyword", "hybrid"]]
print("Rows kept per relevance method (probe = 1 zip/day per election):")
print(rel_summary.to_string())
print()
print("Full-corpus hybrid estimate (×96 since probe stride was 96):")
for k in ELECTIONS:
    print(f"  {k:20s} probe={int(rel_summary.loc[k, 'hybrid']):6d}  →  full-corpus ~{int(rel_summary.loc[k, 'hybrid'])*96:,}")

In [ ]:
"""Precision eyeball: 10 random hybrid-kept rows from each election."""
import random
random.seed(42)
for key in ELECTIONS:
    sub = probe[probe.election == key]
    kept, _ = filter_relevant(sub, ELECTIONS[key], method="hybrid")
    sample = kept.sample(n=min(10, len(kept)), random_state=42)
    print(f"--- {key} hybrid (n_kept={len(kept)}) ---")
    for _, r in sample.iterrows():
        url = (r.DocumentIdentifier or "")[:100]
        print(f"  {r.SourceCommonName:30s}  {url}")
    print()

**What the diagnostic showed.**

- **`theme` alone is election-blind.** It captures every article tagged with `ELECTION`, `GOV_ELECTION`, `POLITICAL_PARTY`, etc. — including elections in other countries during the same window (US 2022 midterms during Kenya 2022, US 2024 primaries during Senegal/SA 2024). High recall on "elections as a topic", low precision on *the named election*. Probe yield: 11–14% per election — far too inclusive.
- **`keyword` alone is election-specific but vulnerable to non-political namesakes.** Word-boundary matching killed the original substring-collision false positives (`avocado`, `bafana`, `effort`), but residual noise remains — sportspeople sharing a candidate's surname, US politicians sharing a party initialism, etc.
- **`hybrid` (theme AND keyword) is the right cut for this question.** Probe yield: 41–366 rows per election (0.05–0.5%). Extrapolating ×96 (probe stride = 1 slot/day; full corpus = 96 slots/day) gives an estimated 4,000–35,000 hybrid-kept rows per election in the full corpus — easily large enough for analysis, small enough that LLM classification stays cheap. **Eyeball precision on a 10-row random sample per election was ~85–90%** — the residual false positives are mostly cross-election cross-referencing (e.g., an article about the US election that mentions Kenya in passing).

**Options considered.**

- **(a) `theme` only.** High recall, low precision. Rejected: pollutes the African block with non-African elections from the same window.
- **(b) `keyword` only.** Reasonable precision after word-boundary fix, but vulnerable to namesake noise (sports articles mentioning a politician's surname). Rejected.
- **(c) `hybrid` (theme AND keyword).** Lowest yield, highest precision. **Selected.**

**Decision.** **`hybrid`** with the broadened keyword list in `src/elections_frames/data.py`. Both signals required: the row must carry an election-related theme tag *and* mention an election-specific keyword. The keyword list was iterated once during this diagnostic — initial substring matching produced 40–70% false positives in South Africa and Senegal; the deployed implementation uses word-boundary regex and an expanded keyword set (candidate first names, party long-forms, electoral commission acronyms).

**Sensitivity.** The hybrid yield is the smallest of the three options; recall is *bounded* by what GDELT tagged with election themes — a row that doesn't carry an `ELECTION` theme tag will be missed even if it's about the election. The eval set's class-imbalance diagnostic (Session 5) will surface this if it bites; for now, a corpus of ~4K–35K relevant rows per election is well above what we need for the analysis. The boundary between `theme` (11–14% yield) and `hybrid` (0.05–0.5%) is large, so this isn't a knife-edge decision.

### Decision 3: Deduplication strategy

**Problem.** Wire copy (Reuters, AP, AFP, PA, local equivalents) gets syndicated across dozens of regional outlets — the same article appears under different URLs at different domains. If we don't dedup, the headline analysis double-counts syndicated framing and over-weights wire services. Two layers to handle: **exact-URL duplicates** (same article re-ingested by GDELT through canonicalization differences — `www.` vs `m.`, tracking params, trailing slashes) and **near-duplicate text** (same wire copy under different URLs).

**Diagnostic.** On the relevance-filtered probe (~700 rows), measure: (1) how many URL-canonicalization duplicates exist; (2) at MinHash Jaccard thresholds 0.7 / 0.8 / 0.9 / 0.95, how many near-duplicate clusters form, and what those clusters actually contain. The text input for MinHash is the GKG-metadata snippet (URL title-slug + V21AllNames + V2EnhancedThemes — per Decision Log #1, no live HTML scraping).

In [ ]:
"""Dedup threshold sweep on the relevance-filtered probe."""
from elections_frames.cleaning import deduplicate, build_text_snippet, canonicalize_url

filtered = []
for k in ELECTIONS:
    sub = probe[probe.election == k]
    kept, _ = filter_relevant(sub, ELECTIONS[k], method="hybrid")
    filtered.append(kept)
rel = pd.concat(filtered, ignore_index=True)
rel["text_snippet"] = rel.apply(build_text_snippet, axis=1)
rel["canon_url"] = rel.DocumentIdentifier.fillna("").apply(canonicalize_url)

print(f"Relevance-filtered probe: {len(rel):,} rows")
print(f"After URL canonicalization: {rel.canon_url.nunique():,} unique  (URL dups removed: {len(rel)-rel.canon_url.nunique()})")
print()

sweep = []
for thr in (0.7, 0.8, 0.9, 0.95):
    _, c = deduplicate(rel.copy(), threshold=thr)
    sweep.append({
        "threshold": thr,
        "n_in": c["n_in"],
        "n_kept": c["n_after_text_dedup"],
        "n_text_dups_merged": c["n_text_duplicates"],
        "pct_kept": round(100 * c["n_after_text_dedup"] / c["n_in"], 1),
    })
print("MinHash threshold sweep:")
print(pd.DataFrame(sweep).to_string(index=False))

In [ ]:
"""Eyeball: inspect 3 random multi-row clusters at threshold 0.8."""
from datasketch import MinHash, MinHashLSH
from elections_frames.cleaning import _shingles

thr = 0.8
url_deduped = rel.drop_duplicates("canon_url").reset_index(drop=True)
lsh = MinHashLSH(threshold=thr, num_perm=128)
mhs = {}
for i, text in enumerate(url_deduped.text_snippet):
    m = MinHash(num_perm=128)
    for sh in _shingles(text):
        m.update(sh.encode("utf-8"))
    mhs[i] = m
    lsh.insert(f"r{i}", m)

# Collect multi-row clusters
clusters: list[list[int]] = []
seen: set[int] = set()
for i in range(len(url_deduped)):
    if i in seen:
        continue
    matches = sorted(int(k[1:]) for k in lsh.query(mhs[i]))
    if len(matches) > 1:
        clusters.append(matches)
        seen.update(matches)

print(f"At threshold={thr}: {len(clusters)} multi-row clusters (covering {sum(len(c) for c in clusters)} rows)")
print()
for c in clusters[:3]:
    print(f"--- cluster (size={len(c)}) ---")
    for idx in c[:4]:
        r = url_deduped.iloc[idx]
        print(f"    {r.SourceCommonName:30s}  {(r.DocumentIdentifier or '')[:90]}")
    if len(c) > 4:
        print(f"    ... ({len(c) - 4} more)")
    print()

**What the diagnostic showed.**

- **URL canonicalization alone catches very few duplicates** on the relevance-filtered probe (0 in this sample) — GDELT's URL coverage is wide enough that the same article rarely shows up under multiple URL spellings *within* GDELT's slot-level deduplication.
- **MinHash near-duplicate detection catches real syndication.** At threshold 0.8: 25 clusters covering ~100 rows (≈14% of the relevance-filtered probe). Inspected clusters are unambiguous wire-copy syndication — e.g., 7+ UK regional newspapers carrying identical PA wire copy under URLs sharing the same wire ID (`*/news/national/23319500.*` across `theargus.co.uk`, `hamhigh.co.uk`, `swindonadvertiser.co.uk`, ...).
- **All thresholds catch the same core clusters.** Going from 0.7 → 0.95 only changes the dedup yield by ~3% (109 → 87 rows merged). Eyeballed clusters at every threshold contain true syndications, no false merges.
- **Note: many syndicated clusters are UK politics articles** that passed the relevance filter because the keyword `"Labour Party"` (Nigeria 2023) matched UK Labour-related copy. This is the cross-election contamination flagged in Decision 2 — dedup is still the right action (collapse the wire copy), and the residual UK-politics rows will be caught by the LLM classifier's relevance check at scoring time. Logged as a known limitation.

**Options considered.**

- **(a) URL canonicalization only.** Misses wire syndication entirely. Rejected.
- **(b) URL canonicalization + MinHash @ Jaccard 0.7.** Most aggressive merging. Slight risk of false merges on superficially-similar non-duplicate articles.
- **(c) URL canonicalization + MinHash @ Jaccard 0.8.** Industry-conventional threshold; in this probe matches every clear-cut syndication cluster while leaving non-duplicates alone.
- **(d) URL canonicalization + MinHash @ Jaccard 0.9 or 0.95.** Most conservative; trades some recall for theoretical false-merge safety. In our probe, the safety margin is unused — no observed false merges at 0.7.

**Decision.** **(c) URL canonicalization + MinHash @ Jaccard 0.8** on the GKG-metadata text snippet. The 0.7–0.95 range produced near-identical results on the probe; 0.8 is the middle-of-the-road conventional choice and we follow it. Earliest-publication-wins (sort by `DATE` ascending before dedup) so syndicated clusters collapse to the wire-service first appearance.

**Sensitivity.** Decision is not sensitive — the headline analysis numbers move by ≤3% across the full 0.7 → 0.95 threshold range on the probe. Robustness notebook will rerun the headline figure at 0.7 and 0.9 to confirm at full-corpus scale.

## 5. Structured analytic dataset

`data/processed/articles_clean.parquet` is the canonical output of the Session-3 pipeline (`scripts/run_cleaning.py`). One row per relevance-filtered, English, deduped article. Columns inherited from GDELT's GKG schema; additional columns `election`, `outlet_origin`, `text_snippet` added by the cleaning pipeline. The eval-set candidates (`data/external/eval_set_candidates.parquet`) are drawn from this by stratified sampling, also below.

In [ ]:
"""Load the cleaned analytic dataset; preview shape, schema, per-election counts."""
clean_path = Path("../data/processed/articles_clean.parquet")
if not clean_path.exists():
    print("articles_clean.parquet not yet produced — run `python scripts/run_cleaning.py` from the project root first.")
    clean = pd.DataFrame()
else:
    clean = pd.read_parquet(clean_path)
    print(f"Loaded {len(clean):,} cleaned articles.")
    print()
    print("Per-election counts:")
    counts = clean.groupby(["election", "outlet_origin"]).size().unstack(fill_value=0)
    counts["TOTAL"] = counts.sum(axis=1)
    print(counts.to_string())
    print()
    print("Schema:")
    print(clean.dtypes.to_string())

### Eval-set candidate sampling (stratified)

**Problem.** The LLM classifier must be evaluated against an independent hand-labeled gold set; we need ~200–300 articles drawn so the evaluation reflects the distribution we care about: African vs. International, all four elections, varied positioning around vote day.

**Decision (lightweight; full sampling-strategy block deferred to Session 9 after labeling).** Stratify by **election × outlet_origin × week-around-vote** (5 buckets: pre-4-weeks, pre-2-weeks, vote-week, post-2-weeks, post-4-weeks) → 40 strata. Allocate ~6 articles per stratum to hit the ~250 target. Strata smaller than the per-stratum allocation contribute everything they have; the residual deficit is filled by an unstratified top-up so we always hit the target. Edge labels are folded into International for sampling (matches the headline-analysis attribution choice). Hand-off documentation lives in `docs/labeling_handoff.md`.

**Sensitivity.** Sampling strategy is one of the seven CLAUDE.md-flagged decisions; the full five-part block for it is finalized in Session 9 once the eval set is labeled and we can compute observed class imbalance.

In [ ]:
"""Stratified sample → eval_set_candidates.parquet."""
from elections_frames.cleaning import sample_eval_candidates

candidates_path = Path("../data/external/eval_set_candidates.parquet")

if len(clean) == 0:
    print("Run the pipeline first (`scripts/run_cleaning.py`).")
elif candidates_path.exists():
    candidates = pd.read_parquet(candidates_path)
    print(f"eval_set_candidates.parquet already exists ({len(candidates):,} rows). Delete it to re-sample.")
else:
    candidates = sample_eval_candidates(clean, target_total=250)
    candidates.to_parquet(candidates_path, index=False)
    print(f"Sampled {len(candidates):,} candidates → {candidates_path}")

if candidates_path.exists():
    candidates = pd.read_parquet(candidates_path)
    print()
    print("Stratification check (election × origin × week-bucket):")
    work = candidates.copy()
    if "week_bucket" not in work.columns:
        from elections_frames.cleaning import _week_bucket
        vote_dates = {k: pd.Timestamp(e.date) for k, e in ELECTIONS.items()}
        work["week_bucket"] = work.apply(lambda r: _week_bucket(r["DATE"], vote_dates[r["election"]]), axis=1)
    if "origin_folded" not in work.columns:
        from elections_frames.cleaning import collapse_edge_to_international
        work["origin_folded"] = collapse_edge_to_international(work["outlet_origin"])
    pivot = work.pivot_table(index=["election", "origin_folded"], columns="week_bucket", values="GKGRECORDID", aggfunc="count", fill_value=0)
    print(pivot.to_string())

## 6. Classification

Frames assigned via NVIDIA NIM (`deepseek-v4-pro` primary, `minimax-m2.7` fallback) using the final prompt selected in `04_pipeline_eval.ipynb`. Confidence threshold picked via the precision/coverage decision block in Section 7. Cost log: `data/processed/llm_cost.csv`.

*(Populated in Session 6.)*

### Decision: Confidence threshold for accepting LLM labels in production

Precision floor declared up front; threshold = lowest confidence meeting the floor on the eval set, to maximize coverage. (Session 6.)

## 7. Analysis

Four comparisons (Session 7):

1. Frame distribution by outlet origin, per election — the headline comparison.
2. Frame distribution over time around vote day (rolling window).
3. Frame distribution by outlet within the African block (since 'African' is not a monolith).
4. Cross-election: which frames are election-specific vs. consistent.

## 8. Hero figure

Stacked bar of frame mix, African vs. International, per election. Rendered at 800×800 and saved to `figures/hero.png`. Must be legible at LinkedIn-post thumbnail size.

## 9. Findings

3–5 falsifiable statements, each anchored in a specific number. (Session 7.)

## 10. Limitations

- GDELT coverage bias (whatever is publicly crawlable).
- English-only filter excludes francophone-only Senegal local coverage.
- LLM-as-classifier is an *evaluated* approximation (see eval scores), not ground truth.
- 6-frame taxonomy is one of several reasonable cuts (see taxonomy decision block).
- 'African' is treated as a block at the headline level; intra-African variation surfaced separately.
- Descriptive, not causal.

## 11. Decisions summary table

Populated in Session 9.

| Decision | Chose | Why (anchored in diagnostic) | Sensitivity |
|----------|-------|------------------------------|-------------|
| ... | ... | ... | ... |

## 12. Reproducibility

Install: `pip install -e ".[viz,nlp,llm]"`. Run order: this notebook end-to-end, with `04_pipeline_eval.ipynb` run first if eval results are stale. Wall-clock: dominated by GDELT pull (Session 2) and LLM classification (Session 6) — cost logged in `data/processed/llm_cost.csv`.